In [7]:
import win32com.client
import os
import json
import re
from pathlib import Path


In [8]:
def clean_text(text):
    if not text: return ""
    # Strip the Quark 'soft return' and other control chars
    text = text.replace('\x07', ' ') 
    return re.sub(r'[^\x20-\x7E\n\r\t]', '', text).strip()

In [9]:
def process_with_quark_2016(source_dir, output_root):
    source_root = Path(source_dir).resolve()
    output_root = Path(output_root).resolve()
    pdf_dir = output_root / "archived_pdfs"
    json_dir = output_root / "standardized_json"
    
    for d in [pdf_dir, json_dir]: d.mkdir(parents=True, exist_ok=True)

    # Initialize QuarkXPress
    print("Launching QuarkXPress 2016...")
    try:
        # Attempt 1: Version specific
        quark = win32com.client.Dispatch("QuarkXPress.Application.2016")
    except Exception:
        try:
            # Attempt 2: Version number
            quark = win32com.client.Dispatch("QuarkXPress.Application.12")
        except Exception:
            # Attempt 3: Generic
            quark = win32com.client.Dispatch("QuarkXPress.Application")
    
    quark.Visible = True # Set to True for debugging so you can see if a dialog box pops up

    for file_path in source_root.rglob("*.qxp"):
        print(f"Processing: {file_path.name}")
        
        try:
            # 1. Open Document
            doc = quark.Open(str(file_path))
            
            # 2. Export PDF
            # Note: Export options may vary based on your local PDF styles
            pdf_path = pdf_dir / f"{file_path.stem}.pdf"
            doc.ExportPDF(str(pdf_path))

            # 3. Extract Text for JSON
            json_data = {
                "source_file": str(file_path).replace("\\", "/"),
                "document_name": file_path.name,
                "page_count": doc.Pages.Count,
                "pages": []
            }

            for p_idx in range(1, doc.Pages.Count + 1):
                page = doc.Pages(p_idx)
                page_data = {"page_name": str(p_idx), "frames": []}
                
                # Iterate through Text Boxes
                for tb_idx in range(1, page.TextBoxes.Count + 1):
                    box = page.TextBoxes(tb_idx)
                    raw_text = box.Story.Text
                    cleaned = clean_text(raw_text)
                    
                    if cleaned:
                        page_data["frames"].append({
                            "text": cleaned,
                            "paragraphs": [{"style": "QuarkBox", "text": cleaned}]
                        })
                
                json_data["pages"].append(page_data)

            # 4. Save JSON
            with open(json_dir / f"{file_path.stem}.json", "w", encoding="utf-8") as f:
                json.dump(json_data, f, indent=4)

            doc.Close(2) # 2 = Don't save changes
            print(f"   Success: {file_path.stem}")

        except Exception as e:
            print(f"   Error processing {file_path.name}: {e}")

    quark.Quit()


In [10]:
if __name__ == "__main__":
    SOURCE = r"E:\Callproject\assembled2\1_05-02 Call\Classifieds\Child Care"
    OUTPUT = r"E:\Callproject\Database_Ready_Native"
    process_with_quark_2016(SOURCE, OUTPUT)

Launching QuarkXPress 2016...


com_error: (-2147221005, 'Invalid class string', None, None)

In [11]:
import win32com.client
import pythoncom

try:
    # We use the generic string first
    quark = win32com.client.Dispatch("QuarkXPress.Application")
    print("Success! Python can now communicate with QuarkXPress 2016.")
    quark.Visible = True
except Exception as e:
    print(f"Connection still failing: {e}")
    print("Checking for version-specific registration...")
    try:
        quark = win32com.client.Dispatch("QuarkXPress.Application.2016")
        print("Success using version-specific string!")
    except Exception as e2:
        print(f"Both attempts failed: {e2}")

Connection still failing: (-2147221005, 'Invalid class string', None, None)
Checking for version-specific registration...
Both attempts failed: (-2147221005, 'Invalid class string', None, None)


In [12]:
import win32com.client
try:
    # Try the Project string which is common in newer versions
    quark = win32com.client.Dispatch("QuarkXPress.Project")
    print("Success! Connected via QuarkXPress.Project")
    quark.Visible = True
except Exception as e:
    print(f"Failed: {e}")

Failed: (-2147221005, 'Invalid class string', None, None)
